# Imports

In [ ]:
import os

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DateType,
    FloatType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

from rich import print

# Init Spark


In [24]:
spark = SparkSession.builder \
    .appName("NCEI_Weather_Analysis") \
    .master("local") \
    .getOrCreate()
# local[*] for all cores
spark.sparkContext.setLogLevel("ERROR")

print("[green](*)[/green] Spark is running!")

(*) Spark is running!

# Data

- Get Weather Data from Cincinnati & Florida from [NCEI.nooa.gov](https://www.ncei.noaa.gov/data/global-summary-of-the-day/access/)
- Required data:
    - Cincinnati (72429793812) from 2015 - 2024
    - Florida (99495199999) from 2015 - 2024 (note that 2016 Florida data does not exist)

In [11]:
# script to download the required data
def download_weather_data(city_code, start_year, end_year, output_dir):
    """
    Downloads weather data for a given city and time range, and saves it to the specified output directory.

    Args:
        city_code (str): The code of the city for which to download weather data.
        start_year (int): The starting year of the data to download.
        end_year (int): The ending year of the data to download.
        output_dir (str): The directory where the downloaded data should be saved.
    
    Returns:
        None
    """

    # make sure output directory exists
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    for year in range(start_year, end_year + 1):
        # construct the URL of ncei.noaa.gov for the specified city and time range
        url = f"https://www.ncei.noaa.gov/data/global-summary-of-the-day/access/{year}/{city_code}.csv"
        # download the data and save it to the output directory
        sub_dir = os.path.join(output_dir, f"year={year}")
        if not os.path.exists(sub_dir):
            os.makedirs(sub_dir)
        output_file = os.path.join(sub_dir, f"{city_code}.csv")
        try:
            # -f flag tells curl to return an error code on 404
            exit_code = os.system(f"curl -f -o {output_file} {url} --silent")   
            
            if exit_code == 0:
                print(f"[green](*)[/green] Data for {city_code} in {year} has been downloaded and saved to {output_file}.")
            else:
                print(f"[red](!)[/red] Failed to download data for {city_code} in {year}. (Likely 404 Not Found)")
                # remove the empty file curl might have created before failing
                if os.path.exists(output_file):
                    os.remove(output_file)
        except Exception as e:
            print(f"[red](!)[/red] An error occurred while downloading data for {city_code} in {year}: {e}")

In [12]:
# run the script to download the data for the specified city and time range
CINCINNATI_WEATHER_CODE = "72429793812"
FLORIDA_WEATHER_CODE = "99495199999"
START_YEAR = 2015
END_YEAR = 2024

download_weather_data(CINCINNATI_WEATHER_CODE, START_YEAR, END_YEAR, "data/")
download_weather_data(FLORIDA_WEATHER_CODE, START_YEAR, END_YEAR, "data/")

(*) Data for 72429793812 in 2015 has been downloaded and saved to data/year=2015/72429793812.csv.

(*) Data for 72429793812 in 2016 has been downloaded and saved to data/year=2016/72429793812.csv.

(*) Data for 72429793812 in 2017 has been downloaded and saved to data/year=2017/72429793812.csv.

(*) Data for 72429793812 in 2018 has been downloaded and saved to data/year=2018/72429793812.csv.

(*) Data for 72429793812 in 2019 has been downloaded and saved to data/year=2019/72429793812.csv.

...

(*) Data for 99495199999 in 2020 has been downloaded and saved to data/year=2020/99495199999.csv.

(*) Data for 99495199999 in 2021 has been downloaded and saved to data/year=2021/99495199999.csv.

(*) Data for 99495199999 in 2022 has been downloaded and saved to data/year=2022/99495199999.csv.

(*) Data for 99495199999 in 2023 has been downloaded and saved to data/year=2023/99495199999.csv.

(*) Data for 99495199999 in 2024 has been downloaded and saved to data/year=2024/99495199999.csv.

## Describe data

In [14]:
# load a sample of the data into a Spark DataFrame to verify that it was downloaded correctly
sample_file_path = "data/year=2015/72429793812.csv"
weather_df = spark.read.csv(sample_file_path, header=True, inferSchema=True)
weather_df.show(5)

# show the schema of the DataFrame to verify that the data types were inferred correctly
weather_df.printSchema()


+-----------+----------+--------+---------+---------+--------------------+----+---------------+----+---------------+------+--------------+-----+--------------+-----+----------------+----+---------------+-----+-----+----+--------------+----+--------------+----+---------------+-----+------+
|    STATION|      DATE|LATITUDE|LONGITUDE|ELEVATION|                NAME|TEMP|TEMP_ATTRIBUTES|DEWP|DEWP_ATTRIBUTES|   SLP|SLP_ATTRIBUTES|  STP|STP_ATTRIBUTES|VISIB|VISIB_ATTRIBUTES|WDSP|WDSP_ATTRIBUTES|MXSPD| GUST| MAX|MAX_ATTRIBUTES| MIN|MIN_ATTRIBUTES|PRCP|PRCP_ATTRIBUTES| SNDP|FRSHTT|
+-----------+----------+--------+---------+---------+--------------------+----+---------------+----+---------------+------+--------------+-----+--------------+-----+----------------+----+---------------+-----+-----+----+--------------+----+--------------+----+---------------+-----+------+
|72429793812|2015-01-01|  39.106|-84.41609|    144.8|CINCINNATI MUNICI...|26.1|             24|10.9|             24|1025.5|       

In [15]:
weather_df.describe().show()

+-------+---------------+-----------------+------------------+------------------+--------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+-----------------+--------------+------------------+--------------+-------------------+---------------+-----------------+------------------+
|summary|        STATION|         LATITUDE|         LONGITUDE|         ELEVATION|                NAME|              TEMP|    TEMP_ATTRIBUTES|              DEWP|   DEWP_ATTRIBUTES|               SLP|    SLP_ATTRIBUTES|               STP|    STP_ATTRIBUTES|             VISIB|   VISIB_ATTRIBUTES|              WDSP|   WDSP_ATTRIBUTES|             MXSPD|              GUST|              MAX|MAX_ATTRIBUTES|               MIN|MIN_ATTRIBUTES|               PRCP|PRCP_ATTRIBUTES|             SND

## Load data

In [34]:
# load data from the downloaded files into a Spark DataFrame
# partition discovery via .option("basePath", "data/")
cincinnati_df = spark.read \
    .option("basePath", "data/") \
    .csv(f"data/*/{CINCINNATI_WEATHER_CODE}.csv", header=True, inferSchema=True)

florida_df = spark.read \
    .option("basePath", "data/") \
    .csv(f"data/*/{FLORIDA_WEATHER_CODE}.csv", header=True, inferSchema=True)

weather_df=spark.read.csv("data/", header=True, inferSchema=True)
# summary
print(f"[green](*)[/green] Cincinnati weather data has {cincinnati_df.count()} records.")
print(f"[green](*)[/green] Florida weather data has {florida_df.count()} records.")
print(f"[green](*)[/green] Weather data has {weather_df.count()} records.")

(*) Cincinnati weather data has 3653 records.

(*) Florida weather data has 2483 records.

(*) Weather data has 6136 records.

In [35]:
cincinnati_df.columns

['STATION',
 'DATE',
 'LATITUDE',
 'LONGITUDE',
 'ELEVATION',
 'NAME',
 'TEMP',
 'TEMP_ATTRIBUTES',
 'DEWP',
 'DEWP_ATTRIBUTES',
 'SLP',
 'SLP_ATTRIBUTES',
 'STP',
 'STP_ATTRIBUTES',
 'VISIB',
 'VISIB_ATTRIBUTES',
 'WDSP',
 'WDSP_ATTRIBUTES',
 'MXSPD',
 'GUST',
 'MAX',
 'MAX_ATTRIBUTES',
 'MIN',
 'MIN_ATTRIBUTES',
 'PRCP',
 'PRCP_ATTRIBUTES',
 'SNDP',
 'FRSHTT',
 'year']

## Cleanup

In [36]:
# missing data to n/a
null_9999 = ["TEMP", "DEWP", "SLP", "STP", "MAX", "MIN"]
null_99   = ["PRCP"]
null_999  = ["SNDP", "VISIB", "WDSP", "MXSPD", "GUST"]

clean_expressions = []

for col_name in florida_df.columns:
    if col_name in null_9999:
        # If it's a 9999.9 column, replace 9999.9 with None
        expr = F.when(F.col(col_name) == 9999.9, None).otherwise(F.col(col_name)).alias(col_name)
    elif col_name in null_99:
        # If it's PRCP, replace 99.99 with None
        expr = F.when(F.col(col_name) == 99.99, None).otherwise(F.col(col_name)).alias(col_name)
    elif col_name in null_999:
        # If it's Snow Depth, replace 999.9 with None
        expr = F.when(F.col(col_name).isin([999.9, 999, 999.0]), None).otherwise(F.col(col_name)).alias(col_name)
    else:
        # If it's a normal column (like STN, YEAR, DATE), just keep it as is
        expr = F.col(col_name)
        
    clean_expressions.append(expr)

cleaned_cincinnati_df = cincinnati_df.select(*clean_expressions)
cleaned_florida_df = florida_df.select(*clean_expressions)
cleaned_weather_df = weather_df.select(*clean_expressions)

In [37]:
cleaned_florida_df.select("TEMP", "PRCP", "MAX").show(5)

+----+----+----+
|TEMP|PRCP| MAX|
+----+----+----+
|61.7| 0.0|65.7|
|69.3| 0.0|74.1|
|73.4| 0.0|80.6|
|74.2| 0.0|81.5|
|58.2| 0.0|67.3|
+----+----+----+
only showing top 5 rows


# Analysis


#### **3. Find the hottest day (column MAX) for each year (Points: 1):**
> *Provide the corresponding station code, station name, date, and temperature (columns: `STATION`, `NAME`, `DATE`, `MAX`). There should be 10 results.*

In [38]:
# # hottest date in weather df 
# hottest_date = weather_df.orderBy(F.desc("TMAX")).first()["DATE"]
# print(f"[green](*)[/green] The hottest date in the weather data is {hottest_date}.")

#### **4. Find the coldest day (column MIN) for the month of March across all years (2015-2024) (Points: 1):**
> *Provide the corresponding station code, station name, date, and temperature (columns: `STATION`, `NAME`, `DATE`, `MIN`). There should be 1 result.*

#### **5. Find the year with the most precipitation for Cincinnati and Florida (Points: 1):**
> *Provide the corresponding station code, station name, and year (columns: `STATION`, `NAME`, `YEAR`, Mean of `PRCP`). There should be 2 results.*

#### **6. Count the percentage of missing values for wind gust (column GUST) for Cincinnati and Florida in the year 2024 (Points: 1):**
> *There should be 2 results.*

#### **7. Find the mean, median, mode, and standard deviation of the temperature (column TEMP) for Cincinnati in each month for the year 2020 (Points: 1):**
> *There should be 12 results (one for each month, with 4 values for each result).*

#### **8. Find the top 10 days with the lowest Wind Chill for Cincinnati in 2017 (Points: 1):**
> For days where `TEMP` is below 50°F and `WDSP` (wind speed) is above 3 mph, calculate Wind Chill using the formula:
`WC = 35.74 + 0.6215 × TEMP − 35.75 × (WDSP)^0.16 + 0.4275 × TEMP × (WDSP)^0.16`
Add a new column for Wind Chill and display the top 10 days with the lowest Wind Chill.

#### **9. Investigate how many days had extreme weather conditions for Florida (fog, rain, snow, etc.) using the `FRSHTT` column (Points: 1).**
> *There should be 1 result. Details can be found in the [README.pdf](https://www.ncei.noaa.gov/data/global-summary-of-the-day/doc/readme.pdf).*

#### **10. Predict the maximum Temperature for Cincinnati for November and December 2024, based on the previous 2 years of weather data (Points: 1):**
> There should be 2 results.
You can use any model  to forecast on the historical weather data.
Submit the code, prediction results, and a brief discussion on the model’s performance and potential improvements.

In [39]:
# Stop the session when ompletely done with the notebook
# spark.stop()